In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. CREATE SYNTHETIC DATA (Mimicking Sus-Pashan Road patterns)
np.random.seed(42)
data_size = 1000

# Generating random hours (0-23), months (1-12), and day types (0=Weekend, 1=Weekday)
hours = np.random.randint(0, 24, data_size)
months = np.random.randint(1, 13, data_size)
is_weekday = np.random.randint(0, 2, data_size)
congestion = []

# Logic to mimic local traffic patterns
for h, m, w in zip(hours, months, is_weekday):
    # Morning (8-10) and Evening (17-20) rush on weekdays
    if w == 1 and ((8 <= h <= 10) or (17 <= h <= 20)):
        level = 'High'
    # Monsoon season (Jun-Sep) adds baseline congestion
    elif 6 <= m <= 9 and (7 <= h <= 21):
        level = np.random.choice(['Medium', 'High'])
    # Summer break (Apr-May) or late night / early morning
    elif 4 <= m <= 5 or h < 7 or h > 21:
        level = 'Low'
    # Weekends are generally flatter
    elif w == 0:
        level = np.random.choice(['Low', 'Medium'], p=[0.7, 0.3])
    else:
        level = 'Medium'
    congestion.append(level)

# Create a DataFrame
df = pd.DataFrame({'Hour': hours, 'Month': months, 'Is_Weekday': is_weekday, 'Congestion_Level': congestion})

# 2. PREPARE THE DATA
# Features (X) and Target (y)
X = df[['Hour', 'Month', 'Is_Weekday']]
y = df['Congestion_Level']

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. TRAIN THE MODEL
# Initialize the Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the training data
model.fit(X_train, y_train)

# 4. EVALUATE THE MODEL
# Make predictions on the test set
predictions = model.predict(X_test)

# Calculate and print accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

# 5. TEST WITH A CUSTOM PREDICTION
# Predicting a specific scenario: Weekday, 9 AM, August (Monsoon)
sample_prediction = model.predict([[9, 8, 1]])
print(f"Prediction for Weekday, 9 AM in August: {sample_prediction[0]}")

Model Accuracy: 85.50%

Classification Report:
              precision    recall  f1-score   support

        High       0.88      0.84      0.86        44
         Low       0.94      0.90      0.92       113
      Medium       0.64      0.74      0.69        43

    accuracy                           0.85       200
   macro avg       0.82      0.83      0.82       200
weighted avg       0.87      0.85      0.86       200

Prediction for Weekday, 9 AM in August: High


c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [4]:
import pandas as pd
import numpy as np

# 1. SET UP THE TIMELINE (2 Years of Hourly Data)
print("Generating 2 years of hourly timestamps...")
# Creates a row for every hour from Jan 1, 2024 to Dec 31, 2025
dates = pd.date_range(start='2024-01-01', end='2025-12-31 23:00:00', freq='h')
df = pd.DataFrame({'Datetime': dates})

# Extract useful features for the model
df['Year'] = df['Datetime'].dt.year
df['Month'] = df['Datetime'].dt.month
df['Hour'] = df['Datetime'].dt.hour
df['Day_of_Week'] = df['Datetime'].dt.dayofweek
df['Is_Weekend'] = df['Day_of_Week'].isin([5, 6]).astype(int)

# 2. DEFINE THE TRAFFIC RULES (Based on your graphs)
def calculate_traffic(row):
    # Base hourly volume increases year over year (from your 2023-2025 graph)
    base_volume = 580 if row['Year'] == 2024 else 680
    
    # --- Seasonality Multiplier (From your Seasonal Traffic graph) ---
    if row['Month'] in [4, 5]:  # Summer Break (April-May dip)
        season_mult = 0.85
    elif row['Month'] in [6, 7, 8, 9]:  # Monsoon Season (High congestion)
        season_mult = 1.35
    else:
        season_mult = 1.0  # Standard months
        
    # --- Time-of-Day & Weekend Multiplier (From your Daily Timing graph) ---
    if row['Is_Weekend'] == 1:
        # Weekends hover at a flat, lower rate (~450-500 vehicles)
        time_mult = 0.8
    else:
        # Weekday Rush Hours
        if 8 <= row['Hour'] <= 10:    # Morning Commuter Rush
            time_mult = 2.4
        elif 17 <= row['Hour'] <= 20: # Evening Commuter Rush
            time_mult = 2.6
        elif 0 <= row['Hour'] <= 5:   # Late Night / Early Morning
            time_mult = 0.6
        else:                         # Standard Daytime Traffic
            time_mult = 1.1

    # Calculate raw volume and add random noise for realism
    raw_volume = base_volume * season_mult * time_mult
    noise = np.random.normal(0, 45) # +/- 45 cars of random variance
    final_volume = int(max(0, raw_volume + noise))
    
    return final_volume

print("Applying Sus-Pashan traffic patterns...")
df['Traffic_Volume'] = df.apply(calculate_traffic, axis=1)

# 3. ASSIGN CONGESTION LABELS
def classify_congestion(vol):
    if vol < 800:
        return 'Low'
    elif vol < 1350:
        return 'Medium'
    else:
        return 'High'

df['Congestion_Level'] = df['Traffic_Volume'].apply(classify_congestion)

# 4. EXPORT TO CSV
file_name = 'Sus_Pashan_Traffic_2024_2025.csv'
df.to_csv(file_name, index=False)
print(f"Success! Dataset saved as: {file_name}")
print(f"Total rows generated: {len(df)}")
df.head(10) # Display the first 10 rows to verify

Generating 2 years of hourly timestamps...
Applying Sus-Pashan traffic patterns...
Success! Dataset saved as: Sus_Pashan_Traffic_2024_2025.csv
Total rows generated: 17544


,Datetime,Year,Month,Hour,Day_of_Week,Is_Weekend,Traffic_Volume,Congestion_Level
0,2024-01-01 00:00:00,2024,1,0,0,0,304,Low
1,2024-01-01 01:00:00,2024,1,1,0,0,347,Low
2,2024-01-01 02:00:00,2024,1,2,0,0,394,Low
3,2024-01-01 03:00:00,2024,1,3,0,0,370,Low
4,2024-01-01 04:00:00,2024,1,4,0,0,406,Low
5,2024-01-01 05:00:00,2024,1,5,0,0,322,Low
6,2024-01-01 06:00:00,2024,1,6,0,0,555,Low
7,2024-01-01 07:00:00,2024,1,7,0,0,647,Low
8,2024-01-01 08:00:00,2024,1,8,0,0,1408,High
9,2024-01-01 09:00:00,2024,1,9,0,0,1453,High


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. LOAD THE REALISTIC DATASET
print("Loading Sus-Pashan traffic data...")
df = pd.read_csv('Sus_Pashan_Traffic_2024_2025.csv')

# 2. PREPARE THE DATA
# Features (X): What the model looks at to make a guess
# Target (y): The congestion level it needs to predict
X = df[['Month', 'Hour', 'Is_Weekend']]
y = df['Congestion_Level']

# Split the 17,544 rows into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. TRAIN THE MODEL
print("Training the Random Forest model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. EVALUATE PERFORMANCE
print("Evaluating model accuracy on unseen data...\n")
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Model Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

# 5. RUN A CUSTOM PRESENTATION SCENARIO
# Let's test a known high-risk time: 
# July (Month=7), 6 PM / 18:00 (Hour=18), Weekday (Is_Weekend=0)
scenario = [[7, 18, 0]]
sample_prediction = model.predict(scenario)
print(f"Predicted Congestion for a July Weekday at 6 PM: {sample_prediction[0]}")

Loading Sus-Pashan traffic data...
Training the Random Forest model...
Evaluating model accuracy on unseen data...

Model Accuracy: 95.95%

Classification Report:
              precision    recall  f1-score   support

        High       0.93      0.96      0.94       654
         Low       0.98      0.99      0.99      2358
      Medium       0.90      0.81      0.85       497

    accuracy                           0.96      3509
   macro avg       0.93      0.92      0.93      3509
weighted avg       0.96      0.96      0.96      3509

Predicted Congestion for a July Weekday at 6 PM: High


c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------
# 1. GENERATE EXPANDED DATA (Including Terrain & Weather)
# ---------------------------------------------------------
print("Generating expanded dataset with Terrain & Accident data...")
np.random.seed(42)
data_size = 2000

# Core Features
hours = np.random.randint(0, 24, data_size)
months = np.random.randint(1, 13, data_size)
is_weekday = np.random.randint(0, 2, data_size)

# NEW FEATURE: Route Type (0 = General Route, 1 = Sunny's World Steep Gradient)
is_steep_route = np.random.randint(0, 2, data_size)

congestion_labels = []
accident_risk_labels = []

for h, m, w, steep in zip(hours, months, is_weekday, is_steep_route):
    # --- CONGESTION LOGIC ---
    if w == 1 and ((8 <= h <= 10) or (17 <= h <= 20)):
        cong = 'High'
    elif 6 <= m <= 9 and (7 <= h <= 21):
        cong = np.random.choice(['Medium', 'High'])
    elif 4 <= m <= 5 or h < 7 or h > 21:
        cong = 'Low'
    elif w == 0:
        cong = np.random.choice(['Low', 'Medium'], p=[0.7, 0.3])
    else:
        cong = 'Medium'
    congestion_labels.append(cong)

    # --- ACCIDENT RISK LOGIC ---
    # Monsoon season on the steep gradient during rush hour is highly volatile
    if steep == 1 and (6 <= m <= 9) and cong == 'High':
        risk = 'Critical'
    # Steep route during standard hours or general route during monsoon rush
    elif (steep == 1 and cong != 'Low') or (steep == 0 and 6 <= m <= 9 and cong == 'High'):
        risk = 'Elevated'
    else:
        risk = 'Normal'
    accident_risk_labels.append(risk)

# Build the expanded DataFrame
df = pd.DataFrame({
    'Hour': hours, 
    'Month': months, 
    'Is_Weekday': is_weekday, 
    'Is_Steep_Route': is_steep_route,
    'Congestion_Level': congestion_labels,
    'Accident_Risk': accident_risk_labels
})

# ---------------------------------------------------------
# 2. PREPARE THE DATA FOR TWO MODELS
# ---------------------------------------------------------
# Features (X) now include the terrain data
X = df[['Hour', 'Month', 'Is_Weekday', 'Is_Steep_Route']]

# We have two targets now!
y_congestion = df['Congestion_Level']
y_accident = df['Accident_Risk']

# Split the data
X_train, X_test, y_train_cong, y_test_cong = train_test_split(X, y_congestion, test_size=0.2, random_state=42)
_, _, y_train_acc, y_test_acc = train_test_split(X, y_accident, test_size=0.2, random_state=42)

# ---------------------------------------------------------
# 3. TRAIN BOTH MODELS
# ---------------------------------------------------------
print("Training Congestion Model...")
model_congestion = RandomForestClassifier(n_estimators=100, random_state=42)
model_congestion.fit(X_train, y_train_cong)

print("Training Accident Risk Model...\n")
model_accident = RandomForestClassifier(n_estimators=100, random_state=42)
model_accident.fit(X_train, y_train_acc)

# Evaluate Both
acc_cong = accuracy_score(y_test_cong, model_congestion.predict(X_test))
acc_accid = accuracy_score(y_test_acc, model_accident.predict(X_test))
print(f"Congestion Model Accuracy: {acc_cong * 100:.2f}%")
print(f"Accident Risk Model Accuracy: {acc_accid * 100:.2f}%\n")

# ---------------------------------------------------------
# 4. TEST THE EXPANDED SYSTEM
# ---------------------------------------------------------
# Scenario: 9 AM (Hour=9), July (Month=7), Weekday (1), Sunny's World Steep Patch (1)
scenario = [[9, 7, 1, 1]]

print("--- LIVE SCENARIO PREDICTION ---")
print(f"Scenario: 9:00 AM, July (Monsoon), Weekday, Sunny's World Steep Gradient")
print(f"Predicted Congestion: {model_congestion.predict(scenario)[0]}")
print(f"Predicted Safety Risk: {model_accident.predict(scenario)[0]}")

Generating expanded dataset with Terrain & Accident data...
Training Congestion Model...
Training Accident Risk Model...

Congestion Model Accuracy: 86.75%
Accident Risk Model Accuracy: 89.75%

--- LIVE SCENARIO PREDICTION ---
Scenario: 9:00 AM, July (Monsoon), Weekday, Sunny's World Steep Gradient
Predicted Congestion: High
Predicted Safety Risk: Critical


c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error

# ---------------------------------------------------------
# 1. GENERATE ADVANCED DATA (Congestion, Risk, Speed, AQI)
# ---------------------------------------------------------
print("Generating comprehensive dataset...")
np.random.seed(42)
data_size = 2000

hours = np.random.randint(0, 24, data_size)
months = np.random.randint(1, 13, data_size)
is_weekday = np.random.randint(0, 2, data_size)
is_steep_route = np.random.randint(0, 2, data_size)

congestion_labels, accident_labels, speeds, aqis = [], [], [], []

for h, m, w, steep in zip(hours, months, is_weekday, is_steep_route):
    # --- CONGESTION & RISK LOGIC ---
    if w == 1 and ((8 <= h <= 10) or (17 <= h <= 20)):
        cong = 'High'
    elif 6 <= m <= 9 and (7 <= h <= 21):
        cong = np.random.choice(['Medium', 'High'])
    elif 4 <= m <= 5 or h < 7 or h > 21:
        cong = 'Low'
    elif w == 0:
        cong = np.random.choice(['Low', 'Medium'], p=[0.7, 0.3])
    else:
        cong = 'Medium'
        
    if steep == 1 and (6 <= m <= 9) and cong == 'High':
        risk = 'Critical'
    elif (steep == 1 and cong != 'Low') or (steep == 0 and 6 <= m <= 9 and cong == 'High'):
        risk = 'Elevated'
    else:
        risk = 'Normal'
        
    # --- SPEED LOGIC (Regression) ---
    # Speeds drop as congestion increases
    if cong == 'High':
        spd = np.random.uniform(5, 15)
    elif cong == 'Medium':
        spd = np.random.uniform(15, 35)
    else:
        spd = np.random.uniform(35, 55)
    
    # The Sunny's World steep patch naturally slows down traffic
    if steep == 1:
        spd *= 0.75 

    # --- AQI LOGIC (Regression) ---
    base_aqi = 60 # Baseline good air
    # Winter inversion traps pollution, Monsoons wash it away
    if m in [11, 12, 1, 2]:
        base_aqi += 45
    elif m in [6, 7, 8, 9]:
        base_aqi -= 25
        
    # Idling cars in high congestion spike the local AQI
    if cong == 'High':
        aqi = base_aqi + np.random.uniform(60, 100)
    elif cong == 'Medium':
        aqi = base_aqi + np.random.uniform(25, 50)
    else:
        aqi = base_aqi + np.random.uniform(0, 15)

    congestion_labels.append(cong)
    accident_labels.append(risk)
    speeds.append(round(spd, 1))
    aqis.append(int(aqi))

df = pd.DataFrame({
    'Hour': hours, 'Month': months, 'Is_Weekday': is_weekday, 'Is_Steep_Route': is_steep_route,
    'Congestion': congestion_labels, 'Risk': accident_labels, 'Speed_kmh': speeds, 'AQI': aqis
})

# ---------------------------------------------------------
# 2. PREPARE DATA FOR 4 TARGETS
# ---------------------------------------------------------
X = df[['Hour', 'Month', 'Is_Weekday', 'Is_Steep_Route']]

# Split Data (80% Train, 20% Test)
X_train, X_test, _, _ = train_test_split(X, df['Congestion'], test_size=0.2, random_state=42)

# Targets
y_train_cong = df.loc[X_train.index, 'Congestion']
y_test_cong  = df.loc[X_test.index, 'Congestion']

y_train_risk = df.loc[X_train.index, 'Risk']
y_test_risk  = df.loc[X_test.index, 'Risk']

y_train_spd  = df.loc[X_train.index, 'Speed_kmh']
y_test_spd   = df.loc[X_test.index, 'Speed_kmh']

y_train_aqi  = df.loc[X_train.index, 'AQI']
y_test_aqi   = df.loc[X_test.index, 'AQI']

# ---------------------------------------------------------
# 3. TRAIN MODELS (Classifiers & Regressors)
# ---------------------------------------------------------
print("Training models...")
m_cong = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_train, y_train_cong)
m_risk = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_train, y_train_risk)

# Using Regressors for exact numbers
m_spd = RandomForestRegressor(n_estimators=50, random_state=42).fit(X_train, y_train_spd)
m_aqi = RandomForestRegressor(n_estimators=50, random_state=42).fit(X_train, y_train_aqi)

# ---------------------------------------------------------
# 4. LIVE DASHBOARD PREDICTION
# ---------------------------------------------------------
# Scenario: 6 PM (18), December (12 - Winter), Weekday (1), Steep Route (1)
scenario = [[18, 12, 1, 1]]

print("\n" + "="*45)
print(" 🚦 ADAPTIVE TRAFFIC ANALYSIS DASHBOARD 🚦")
print("="*45)
print("Scenario: Dec Weekday @ 6:00 PM, Steep Gradient")
print(f"Traffic Volume :  {m_cong.predict(scenario)[0]}")
print(f"Accident Risk  :  {m_risk.predict(scenario)[0]}")
print(f"Avg Speed      :  {m_spd.predict(scenario)[0]:.1f} km/h")
print(f"Est. AQI       :  {int(m_aqi.predict(scenario)[0])} (PM2.5/PM10)")
print("="*45)

# Calculate Error Margins for Regressors
print(f"\nModel Error Margins:")
print(f"Speed Estimate is usually off by +/- {mean_absolute_error(y_test_spd, m_spd.predict(X_test)):.1f} km/h")
print(f"AQI Estimate is usually off by +/- {mean_absolute_error(y_test_aqi, m_aqi.predict(X_test)):.1f} points")

Generating comprehensive dataset...
Training models...

 🚦 ADAPTIVE TRAFFIC ANALYSIS DASHBOARD 🚦
Scenario: Dec Weekday @ 6:00 PM, Steep Gradient
Traffic Volume :  High
Accident Risk  :  Elevated
Avg Speed      :  9.9 km/h
Est. AQI       :  179 (PM2.5/PM10)

Model Error Margins:
Speed Estimate is usually off by +/- 5.6 km/h
AQI Estimate is usually off by +/- 8.8 points


c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [8]:
import pickle

# Save the trained models to files
with open('congestion_model.pkl', 'wb') as file:
    pickle.dump(m_cong, file)
with open('risk_model.pkl', 'wb') as file:
    pickle.dump(m_risk, file)
with open('speed_model.pkl', 'wb') as file:
    pickle.dump(m_spd, file)
with open('aqi_model.pkl', 'wb') as file:
    pickle.dump(m_aqi, file)

print("Models saved successfully!")

Models saved successfully!


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import pickle

print("Generating 5-Variable Dataset (Including Road Blockages)...")
np.random.seed(42)
data_size = 2500

hours = np.random.randint(0, 24, data_size)
months = np.random.randint(1, 13, data_size)
is_weekday = np.random.randint(0, 2, data_size)
is_steep_route = np.random.randint(0, 2, data_size)
# NEW: 5% chance of a random road blockage
is_blocked = np.random.choice([0, 1], p=[0.95, 0.05], size=data_size)

congestion_labels, accident_labels, speeds, aqis = [], [], [], []

for h, m, w, steep, blocked in zip(hours, months, is_weekday, is_steep_route, is_blocked):
    # Standard Logic
    if w == 1 and ((8 <= h <= 10) or (17 <= h <= 20)): cong = 'High'
    elif 6 <= m <= 9 and (7 <= h <= 21): cong = np.random.choice(['Medium', 'High'])
    elif 4 <= m <= 5 or h < 7 or h > 21: cong = 'Low'
    elif w == 0: cong = np.random.choice(['Low', 'Medium'], p=[0.7, 0.3])
    else: cong = 'Medium'
        
    if steep == 1 and (6 <= m <= 9) and cong == 'High': risk = 'Critical'
    elif (steep == 1 and cong != 'Low') or (steep == 0 and 6 <= m <= 9 and cong == 'High'): risk = 'Elevated'
    else: risk = 'Normal'
        
    if cong == 'High': spd = np.random.uniform(5, 15)
    elif cong == 'Medium': spd = np.random.uniform(15, 35)
    else: spd = np.random.uniform(35, 55)
    if steep == 1: spd *= 0.75 

    base_aqi = 60
    if m in [11, 12, 1, 2]: base_aqi += 45
    elif m in [6, 7, 8, 9]: base_aqi -= 25
    if cong == 'High': aqi = base_aqi + np.random.uniform(60, 100)
    elif cong == 'Medium': aqi = base_aqi + np.random.uniform(25, 50)
    else: aqi = base_aqi + np.random.uniform(0, 15)

    # --- THE BLOCKAGE OVERRIDE ---
    if blocked == 1:
        cong = 'High'
        risk = 'Critical'
        spd = np.random.uniform(0, 4) # Traffic is at a standstill
        aqi += 50 # Massive pollution spike from idling cars

    congestion_labels.append(cong)
    accident_labels.append(risk)
    speeds.append(round(spd, 1))
    aqis.append(int(aqi))

# Prepare Data
df = pd.DataFrame({'Hour': hours, 'Month': months, 'Is_Weekday': is_weekday, 
                   'Is_Steep_Route': is_steep_route, 'Is_Blocked': is_blocked,
                   'Congestion': congestion_labels, 'Risk': accident_labels, 'Speed': speeds, 'AQI': aqis})

X = df[['Hour', 'Month', 'Is_Weekday', 'Is_Steep_Route', 'Is_Blocked']]

print("Training Upgraded Models...")
m_cong = RandomForestClassifier(n_estimators=50, random_state=42).fit(X, df['Congestion'])
m_risk = RandomForestClassifier(n_estimators=50, random_state=42).fit(X, df['Risk'])
m_spd = RandomForestRegressor(n_estimators=50, random_state=42).fit(X, df['Speed'])
m_aqi = RandomForestRegressor(n_estimators=50, random_state=42).fit(X, df['AQI'])

# Save new models
with open('congestion_model.pkl', 'wb') as f: pickle.dump(m_cong, f)
with open('risk_model.pkl', 'wb') as f: pickle.dump(m_risk, f)
with open('speed_model.pkl', 'wb') as f: pickle.dump(m_spd, f)
with open('aqi_model.pkl', 'wb') as f: pickle.dump(m_aqi, f)
print("Success! New 5-variable .pkl files saved.")